In [0]:
%sql
USE dbjahan.gold_data;

In [0]:
%sql
SELECT DISTINCT drivmedel_kod, drivmedel_namn
FROM dbjahan.Silver_data.nyreg_bilar
ORDER BY drivmedel_kod;

drivmedel_kod,drivmedel_namn
100,bensin
110,diesel
120,el
130,elhybrid
140,laddhybrid
150,etanol/etanol flexifuel
160,gas/gas flexifuel
190,övriga bränslen


In [0]:
%sql
CREATE OR REPLACE TABLE dbjahan.gold_data.dim_kommun AS
SELECT
  k.kommun_kod,
  k.kommun_namn,
  k.elomrade,
  f.folkmangd,
  b.bilar_i_trafik
FROM dbjahan.silver_data.kommun_elomrade k
LEFT JOIN (
  SELECT kommun_kod, folkmangd
  FROM dbjahan.silver_data.folkmangd
  WHERE ar = (SELECT MAX(ar) FROM dbjahan.silver_data.folkmangd)
) f ON k.kommun_kod = f.kommun_kod
LEFT JOIN (
  SELECT kommun_kod, bilar_i_trafik
  FROM dbjahan.silver_data.bilar_trafik
  WHERE ar = (SELECT MAX(ar) FROM dbjahan.silver_data.bilar_trafik)
) b ON k.kommun_kod = b.kommun_kod;

num_affected_rows,num_inserted_rows


In [0]:
%sql
INSERT OVERWRITE dbjahan.gold_data.dim_kommun
SELECT
  k.kommun_kod,
  k.kommun_namn,
  k.elomrade,
  f.folkmangd,
  t.bilar_i_trafik
FROM dbjahan.Silver_data.kommun_elomrade k
LEFT JOIN dbjahan.Silver_data.folkmangd f
  ON k.kommun_kod = f.kommun_kod
 AND f.ar = (SELECT MAX(ar) FROM dbjahan.Silver_data.folkmangd)
LEFT JOIN dbjahan.Silver_data.bilar_trafik t
  ON k.kommun_kod = t.kommun_kod
 AND t.ar = (SELECT MAX(ar) FROM dbjahan.Silver_data.bilar_trafik)

num_affected_rows,num_inserted_rows
290,290


In [0]:
%sql
SELECT COUNT(*) FROM dbjahan.gold_data.dim_kommun;

SELECT f.kommun_kod, d.kommun_kod
FROM dbjahan.gold_data.fakt_laddplats f
LEFT JOIN dbjahan.gold_data.dim_kommun d
  ON f.kommun_kod = d.kommun_kod
WHERE d.kommun_kod IS NULL
LIMIT 10;

kommun_kod,kommun_kod


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW elbil_manad AS
SELECT
  kommun_kod,
  MAX(kommun_namn) AS kommun_namn,
  ar,
  manad,
  SUM(CASE WHEN drivmedel_kod = '120' THEN antal ELSE 0 END) AS antal_el,
  SUM(CASE WHEN drivmedel_kod = '140' THEN antal ELSE 0 END) AS antal_laddhybrid
FROM dbjahan.Silver_data.nyreg_bilar
WHERE drivmedel_kod IN ('120', '140')
GROUP BY kommun_kod, ar, manad;

In [0]:
%sql

CREATE OR REPLACE TABLE dbjahan.gold_data.fakt_laddplats (
  kommun_kod            STRING,
  kommun_namn           STRING,
  elomrade              STRING,
  el_senaste_12         BIGINT,
  el_foregaende_12      BIGINT,
  el_tillvaxt           DOUBLE,
  laddhybrid_senaste_12 BIGINT,
  el_per_1000_inv       DOUBLE,
  folkmangd             BIGINT,
  elpris                DOUBLE,
  poang                 DOUBLE
);

In [0]:
%sql
INSERT OVERWRITE dbjahan.gold_data.fakt_laddplats
WITH ordnad AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY kommun_kod
      ORDER BY ar DESC, manad DESC
    ) AS nr
  FROM elbil_manad
),
summor AS (
  SELECT
    kommun_kod,
    MAX(kommun_namn) AS kommun_namn,
    SUM(CASE WHEN nr <= 12 THEN antal_el ELSE 0 END) AS el_senaste_12,
    SUM(CASE WHEN nr BETWEEN 13 AND 24 THEN antal_el ELSE 0 END) AS el_foregaende_12,
    SUM(CASE WHEN nr <= 12 THEN antal_laddhybrid ELSE 0 END) AS laddhybrid_senaste_12
  FROM ordnad
  GROUP BY kommun_kod
),
bas AS (
  SELECT
    s.kommun_kod,
    s.kommun_namn,
    d.elomrade,
    s.el_senaste_12,
    s.el_foregaende_12,
    CASE
      WHEN s.el_foregaende_12 > 0
      THEN (s.el_senaste_12 - s.el_foregaende_12) / s.el_foregaende_12
    END AS el_tillvaxt,
    s.laddhybrid_senaste_12,
    CASE
      WHEN d.folkmangd > 0
      THEN s.el_senaste_12 / d.folkmangd * 1000
    END AS el_per_1000_inv,
    d.folkmangd,
    p.snitt_sek_per_kwh AS elpris
  FROM summor s
  LEFT JOIN dbjahan.gold_data.dim_kommun d
    ON s.kommun_kod = d.kommun_kod
  LEFT JOIN dbjahan.silver_data.elpris p
    ON d.elomrade = p.elomrade
   AND p.ar = (SELECT MAX(ar) FROM dbjahan.silver_data.elpris)
)
SELECT
  kommun_kod,
  kommun_namn,
  elomrade,
  el_senaste_12,
  el_foregaende_12,
  el_tillvaxt,
  laddhybrid_senaste_12,
  el_per_1000_inv,
  folkmangd,
  elpris,
  ROUND(
    0.4 * PERCENT_RANK() OVER (ORDER BY COALESCE(el_per_1000_inv, 0))
    + 0.4 * PERCENT_RANK() OVER (ORDER BY COALESCE(el_tillvaxt, 0))
    + 0.2 * PERCENT_RANK() OVER (ORDER BY elpris DESC)
  , 3) AS poang
FROM bas;

num_affected_rows,num_inserted_rows
291,291


In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbjahan.gold_data.topp_50_kommuner AS
SELECT * FROM dbjahan.gold_data.fakt_laddplats
ORDER BY poang DESC
LIMIT 50;

num_affected_rows,num_inserted_rows


In [0]:
%sql
INSERT OVERWRITE dbjahan.gold_data.topp_50_kommuner
SELECT *
FROM dbjahan.gold_data.fakt_laddplats
WHERE el_senaste_12 > 0
ORDER BY poang DESC
LIMIT 50;

num_affected_rows,num_inserted_rows
50,50


In [0]:
%sql
SELECT COUNT(*) FROM dbjahan.gold_data.dim_kommun;
SELECT COUNT(*) FROM dbjahan.gold_data.fakt_laddplats;
SELECT * FROM dbjahan.gold_data.topp_50_kommuner;

kommun_kod,kommun_namn,elomrade,el_senaste_12,el_foregaende_12,el_tillvaxt,laddhybrid_senaste_12,el_per_1000_inv,folkmangd,elpris,poang
1486,Strömstad,SE3,204,90,1.2666666666666666,58,15.131286159323542,13482,0.5557320647299667,0.81
1484,Lysekil,SE3,204,115,0.7739130434782608,69,14.668871791184294,13907,0.5557320647299667,0.775
2505,Arvidsjaur,SE1,35,13,1.6923076923076923,19,5.748070290688126,6089,0.22366247936569844,0.766
1494,Lidköping,SE3,639,383,0.6684073107049608,347,15.80705009276438,40425,0.5557320647299667,0.766
2184,Hudiksvall,SE2,297,199,0.49246231155778897,183,7.914090812193563,37528,0.21845232079027763,0.765
1463,Mark,SE3,402,226,0.7787610619469026,230,11.435073247048784,35155,0.5557320647299667,0.759
1443,Bollebygd,SE3,96,53,0.8113207547169812,31,9.793919608243217,9802,0.5557320647299667,0.746
2281,Sundsvall,SE2,813,582,0.39690721649484534,547,8.20814150714805,99048,0.21845232079027763,0.743
1880,Örebro,SE3,1675,1004,0.6683266932270916,1761,10.459597851879606,160140,0.5557320647299667,0.728
1419,Tjörn,SE3,182,114,0.5964912280701754,66,11.309967685806612,16092,0.5557320647299667,0.723
